In [ ]:
# 02 — 추론 실습
검출 → 분류 → 판정(3회 연속) → 이벤트 기록의 전체 흐름.

In [ ]:
# YOLO 검출 + ROI 크롭 (자재 사진으로 테스트)
from ultralytics import YOLO
m = YOLO("yolov8n.pt")
r = m.predict("/workspace/edge-project/data/sample.jpg", verbose=False)[0]
print("검출:", [(m.names[int(c)], round(float(s), 2)) for c, s in zip(r.boxes.cls, r.boxes.conf)])

In [ ]:
# DINOv2 임베딩 + 어텐션 맵
from transformers import Dinov2Model
from PIL import Image
from scripts.attention_map import dinov2_attention_rollout, overlay
b = Dinov2Model.from_pretrained("facebook/dinov2-small", cache_dir="/workspace/.hf-cache")
heat, _ = dinov2_attention_rollout(b, Image.open("/workspace/edge-project/data/sample.jpg"))
overlay(Image.open("/workspace/edge-project/data/sample.jpg"), heat).save("attention.png")
print("어텐션 맵 저장: attention.png")

In [ ]:
# 판정 상태머신 (3회 연속 확정 + 입출고 이벤트)
from scripts.edge_infer import DecisionBuffer, StockStateMachine, lora_packet
buf = DecisionBuffer(3); stk = StockStateMachine()
print(buf.update(0, True), buf.update(0, True), buf.update(0, True))
ev = stk.update(0, 15)
print("이벤트:", ev, "| LoRa 패킷:", len(lora_packet(1, 1, ev)), "B")